# Shell Growth 3D

Render a reduced 3D growth view containing only sampled aperture ellipses and the shell growth path.

In [ ]:
import numpy as np
import plotly.graph_objects as go

In [ ]:
def shell_growth_ring_count(shell_mesh):
    """
    Infer the number of swept aperture rings in a shell mesh.

    The shell mesh stores face growth indices for each stitched triangle. The
    largest face growth index corresponds to the final aperture ring. This is
    independent of whether inner wall vertices are present later in the vertex
    arrays.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :return: Number of growth rings in the outer shell surface
    """
    *_, face_growth_index = shell_mesh

    if len(face_growth_index) == 0:
        raise ValueError("Cannot infer ring count from a shell mesh without face growth indices")

    return int(np.max(face_growth_index)) + 1

In [ ]:
def aperture_ring(shell_mesh, ring_index, aperture_points):
    """
    Extract one outer aperture ring from a shell mesh.

    The outer shell vertices are stored first as contiguous aperture rings:
    ring 0, ring 1, ring 2, and so on. Optional inner wall vertices are appended
    after these outer rings and are intentionally ignored here.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param ring_index: Growth ring index to extract
    :param aperture_points: Number of vertices around each aperture ring
    :return: Tuple of x, y, z arrays for the selected ring
    """
    X, Y, Z, *_ = shell_mesh
    start = ring_index * aperture_points
    end = start + aperture_points

    return X[start:end], Y[start:end], Z[start:end]

In [ ]:
def aperture_ring_centre(shell_mesh, ring_index, aperture_points):
    """
    Estimate the centre of one aperture ring.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param ring_index: Growth ring index to extract
    :param aperture_points: Number of vertices around each aperture ring
    :return: 3D centre point as a NumPy array
    """
    ring_x, ring_y, ring_z = aperture_ring(shell_mesh, ring_index, aperture_points)

    return np.array([
        np.mean(ring_x),
        np.mean(ring_y),
        np.mean(ring_z)
    ])

In [ ]:
def sampled_aperture_indices(shell_mesh, aperture_points, aperture_interval=25, include_final=True):
    """
    Choose growth ring indices where aperture ellipses should be drawn.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_points: Number of vertices around each aperture ring
    :param aperture_interval: Interval between drawn aperture rings
    :param include_final: Whether to force inclusion of the final aperture
    :return: NumPy array of selected ring indices
    """
    if aperture_interval < 1:
        raise ValueError("aperture_interval must be at least 1")

    ring_count = shell_growth_ring_count(shell_mesh)
    indices = list(range(0, ring_count, aperture_interval))

    if include_final and indices[-1] != ring_count - 1:
        indices.append(ring_count - 1)

    return np.array(indices, dtype=int)

In [ ]:
def create_aperture_growth_trace(
    shell_mesh,
    aperture_points,
    aperture_interval=25,
    include_final=True,
    line_colour="rgba(245, 245, 245, 0.82)",
    line_width=2,
):
    """
    Create a single Plotly line trace containing sampled aperture ellipses.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_points: Number of vertices around each aperture ring
    :param aperture_interval: Interval between drawn aperture rings
    :param include_final: Whether to force inclusion of the final aperture
    :param line_colour: Plotly line colour for aperture rings
    :param line_width: Plotly line width for aperture rings
    :return: Plotly Scatter3d trace
    """
    xs, ys, zs = [], [], []
    indices = sampled_aperture_indices(
        shell_mesh,
        aperture_points,
        aperture_interval=aperture_interval,
        include_final=include_final,
    )

    for ring_index in indices:
        ring_x, ring_y, ring_z = aperture_ring(shell_mesh, ring_index, aperture_points)

        xs.extend(np.append(ring_x, ring_x[0]))
        ys.extend(np.append(ring_y, ring_y[0]))
        zs.extend(np.append(ring_z, ring_z[0]))
        xs.append(None)
        ys.append(None)
        zs.append(None)

    return go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="lines",
        name="Apertures",
        line=dict(color=line_colour, width=line_width),
        hoverinfo="skip",
    )

In [ ]:
def create_growth_path_trace(
    shell_mesh,
    aperture_points,
    line_colour="rgb(255, 193, 79)",
    line_width=5,
):
    """
    Create a Plotly line trace through the centre of every aperture ring.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_points: Number of vertices around each aperture ring
    :param line_colour: Plotly line colour for the growth path
    :param line_width: Plotly line width for the growth path
    :return: Plotly Scatter3d trace
    """
    ring_count = shell_growth_ring_count(shell_mesh)
    centres = np.array([
        aperture_ring_centre(shell_mesh, ring_index, aperture_points)
        for ring_index in range(ring_count)
    ])

    return go.Scatter3d(
        x=centres[:, 0],
        y=centres[:, 1],
        z=centres[:, 2],
        mode="lines",
        name="Growth path",
        line=dict(color=line_colour, width=line_width),
        hoverinfo="skip",
    )

In [ ]:
def shell_growth_3d_plotting_options(options, plotting_mode="opaque", title=None):
    """
    Derive Plotly layout options for the aperture-growth view from a preset.

    :param options: Loaded shell preset dictionary
    :param plotting_mode: Plotting options block to use as the layout base
    :param title: Optional figure title override
    :return: Dictionary compatible with apply_shell_layout
    """
    plot_options = dict(options["plotting"][plotting_mode])
    plot_options["title"] = title or f"{options['name']} Growth Path and Apertures"

    return plot_options

In [ ]:
def create_shell_growth_3d_figure(
    options,
    shell_mesh,
    aperture_interval=25,
    include_final=True,
    viewpoint="iso",
    zoom=1.0,
    plotting_mode="opaque",
    title=None,
):
    """
    Build a reduced 3D figure showing only sampled apertures and growth path.

    :param options: Loaded shell preset dictionary
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_interval: Interval between drawn aperture rings
    :param include_final: Whether to force inclusion of the final aperture
    :param viewpoint: Named viewpoint, or a Plotly camera eye dictionary
    :param zoom: Zoom factor; smaller zooms in
    :param plotting_mode: Plotting options block to use as the layout base
    :param title: Optional figure title override
    :return: Plotly Figure
    """
    aperture_points = options["shell"]["aperture_points"]
    plot_options = shell_growth_3d_plotting_options(
        options,
        plotting_mode=plotting_mode,
        title=title,
    )

    fig = go.Figure(data=[
        create_aperture_growth_trace(
            shell_mesh,
            aperture_points,
            aperture_interval=aperture_interval,
            include_final=include_final,
        ),
        create_growth_path_trace(
            shell_mesh,
            aperture_points,
        ),
    ])

    apply_shell_layout(fig, plot_options)

    if isinstance(viewpoint, str):
        if viewpoint not in VIEWPOINTS:
            raise ValueError(
                f"Unknown viewpoint '{viewpoint}'. "
                f"Available viewpoints: {list(VIEWPOINTS.keys())}"
            )
        camera_eye = VIEWPOINTS[viewpoint]
    else:
        camera_eye = viewpoint

    scaled_camera_eye = {k: v * zoom for k, v in camera_eye.items()}

    fig.update_layout(
        scene_camera=dict(eye=scaled_camera_eye),
        showlegend=False,
    )

    return fig

In [ ]:
def render_shell_growth_3d(
    options,
    shell_mesh,
    aperture_interval=25,
    include_final=True,
    viewpoint="iso",
    zoom=1.0,
    plotting_mode="opaque"
):
    """
    Render the 3D aperture-growth view for a shell mesh

    :param options: Preset rendering options
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_interval: Interval between drawn aperture rings
    :param include_final: Whether to force inclusion of the final aperture
    :param viewpoint: Named viewpoint, or a Plotly camera eye dictionary
    :param zoom: Zoom factor; smaller zooms in
    :param plotting_mode: Plotting options block to use as the layout base
    :return: Plotly Figure
    """
    fig = create_shell_growth_3d_figure(
        options,
        shell_mesh,
        aperture_interval=aperture_interval,
        include_final=include_final,
        viewpoint=viewpoint,
        zoom=zoom,
        plotting_mode=plotting_mode,
    )

    fig.show()

    return fig